# Elliptic Data Set: Models (ROC-AUC 94.63)

A benchmark of classical, ensemble and graph models for illicit Bitcoin
transaction detection on the Elliptic dataset, all evaluated the same way.

## Setup

- **Temporal split.** Train on time steps 1 to 34, test on 35 to 49 (the
  split from the original Elliptic paper). A random split would put
  transactions from the same fraud campaign on both sides and inflate
  scores, so it is deliberately not used.
- **Features.** The 165 raw features plus the 1-hop neighbor mean of every
  feature, node degree, and a 3-step rolling average of per-time-step
  feature means (341 features total).
- **Class imbalance.** Balanced class weights or `scale_pos_weight` for
  every model that supports it.
- **Metrics.** ROC-AUC, ROC-PR (average precision) and F1 for the illicit
  class, computed only on the held-out test period.

## Models

1. Gaussian Naive Bayes (baseline)
2. XGBoost, non-tuned and Optuna-tuned (time-series CV)
3. Random Forest, LightGBM, CatBoost (non-tuned)
4. Formula-weighted ensemble of XGBoost, LightGBM, CatBoost and Random Forest
5. Linear stacking ensemble over the same four base learners
6. GraphSAGE + attention GNN, 3 hops (full-batch, GPU)

In [ ]:
import os
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import torch
import torch.nn.functional as F
import xgboost as xgb
from catboost import CatBoostClassifier
from scipy.optimize import minimize
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, classification_report,
                             f1_score, roc_auc_score)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler

try:
    import torch_geometric  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, SAGEConv

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
N_JOBS = os.cpu_count() or 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS = {}
print("cpus:", N_JOBS, "device:", DEVICE)

## Load data and build graph-aware features

In [ ]:
def find_data_dir(filename="elliptic_txs_features.csv", root="/kaggle/input"):
    for dirpath, _, filenames in os.walk(root):
        if filename in filenames:
            return dirpath
    raise FileNotFoundError(f"Could not locate {filename} under {root}")


DATA_DIR = find_data_dir()
FEAT_COLS = [f"local_{i}" for i in range(93)] + [f"agg_{i}" for i in range(72)]

feats = pd.read_csv(f"{DATA_DIR}/elliptic_txs_features.csv", header=None)
feats.columns = ["txId", "time_step"] + FEAT_COLS
classes = pd.read_csv(f"{DATA_DIR}/elliptic_txs_classes.csv")
edges = pd.read_csv(f"{DATA_DIR}/elliptic_txs_edgelist.csv")

df = feats.merge(classes, on="txId", how="left")
df["label"] = df["class"].astype(str).map({"1": 1, "2": 0, "unknown": -1})
df = df.reset_index(drop=True)
id2idx = dict(zip(df["txId"], np.arange(len(df))))
src, dst = edges["txId1"].map(id2idx), edges["txId2"].map(id2idx)
valid = src.notna() & dst.notna()
edge_index = np.vstack([src[valid].to_numpy(), dst[valid].to_numpy()]).astype(np.int64)


def build_graph_features(df, edge_index):
    n = len(df)
    feat = df[FEAT_COLS].to_numpy(dtype=np.float32)
    s, d = edge_index
    deg = np.zeros(n, dtype=np.float32)
    neigh_sum = np.zeros((n, feat.shape[1]), dtype=np.float32)
    np.add.at(deg, s, 1.0)
    np.add.at(deg, d, 1.0)
    np.add.at(neigh_sum, s, feat[d])
    np.add.at(neigh_sum, d, feat[s])
    neigh_mean = neigh_sum / np.where(deg == 0, 1.0, deg)[:, None]
    ts_mean = df.groupby("time_step")[FEAT_COLS[:10]].mean().rolling(3, min_periods=1).mean()
    roll = ts_mean.reindex(df["time_step"].to_numpy()).to_numpy(dtype=np.float32)
    return np.concatenate([feat, neigh_mean, deg[:, None], roll], axis=1)


X = build_graph_features(df, edge_index)
y = df["label"].to_numpy()
labeled = y != -1
train_mask = (df["time_step"] <= 34).to_numpy() & labeled
test_mask = (df["time_step"] > 34).to_numpy() & labeled
Xtr, ytr, Xte, yte = X[train_mask], y[train_mask], X[test_mask], y[test_mask]
ts_train = df["time_step"].to_numpy()[train_mask]
spw = float((ytr == 0).sum() / (ytr == 1).sum())
print("features", X.shape[1], "train", len(ytr), "test", len(yte), "scale_pos_weight", round(spw, 2))


def score(name, probs, seconds, threshold=0.5):
    pred = (probs >= threshold).astype(int)
    RESULTS[name] = {
        "ROC-AUC": roc_auc_score(yte, probs),
        "ROC-PR": average_precision_score(yte, probs),
        "F1": f1_score(yte, pred),
        "train_s": seconds,
    }
    print(f"{name}: " + ", ".join(f"{k}={v:.4f}" for k, v in RESULTS[name].items()))
    print(classification_report(yte, pred, digits=4))

## 1. Naive Bayes baseline

In [ ]:
t0 = time.time()
sc = StandardScaler().fit(Xtr)
nb = GaussianNB().fit(sc.transform(Xtr), ytr)
score("Naive Bayes (baseline)", nb.predict_proba(sc.transform(Xte))[:, 1], time.time() - t0)

## 2. Tree models and the base learners for the ensembles

XGBoost, LightGBM, CatBoost and Random Forest are fit once on the full
training set. Their scores are the non-tuned results, and the same fitted
models feed the formula and stacking ensembles below.

In [ ]:
def make_base_models():
    return {
        "xgb": xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                 tree_method="hist", n_jobs=N_JOBS,
                                 scale_pos_weight=spw, eval_metric="aucpr"),
        "lgbm": lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1, n_jobs=N_JOBS,
                                   scale_pos_weight=spw, verbosity=-1),
        "cat": CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1,
                                  thread_count=N_JOBS, scale_pos_weight=spw,
                                  verbose=False, random_seed=0),
        "rf": RandomForestClassifier(n_estimators=300, n_jobs=N_JOBS,
                                     class_weight="balanced", random_state=0),
    }


NAMES = {"xgb": "XGBoost (non-tuned)", "lgbm": "LightGBM (non-tuned)",
         "cat": "CatBoost (non-tuned)", "rf": "Random Forest (non-tuned)"}
base = make_base_models()
test_probs, fit_seconds = {}, {}
for key, clf in base.items():
    t0 = time.time()
    clf.fit(Xtr, ytr)
    fit_seconds[key] = time.time() - t0
    test_probs[key] = clf.predict_proba(Xte)[:, 1]
    score(NAMES[key], test_probs[key], fit_seconds[key])

## 3. XGBoost tuned with Optuna

The objective is mean average precision over 4 expanding-window time-series
CV folds. The trial count is kept modest so the notebook fits comfortably in
a Kaggle session.

In [ ]:
uniq = np.sort(np.unique(ts_train))
folds = [(uniq[a], uniq[b]) for a, b in TimeSeriesSplit(n_splits=4).split(uniq)]


def objective(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 100, 500),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        tree_method="hist", n_jobs=N_JOBS, scale_pos_weight=spw, eval_metric="aucpr")
    aps = []
    for tr_steps, va_steps in folds:
        tr, va = np.isin(ts_train, tr_steps), np.isin(ts_train, va_steps)
        m = xgb.XGBClassifier(**params).fit(Xtr[tr], ytr[tr])
        aps.append(average_precision_score(ytr[va], m.predict_proba(Xtr[va])[:, 1]))
    return float(np.mean(aps))


t0 = time.time()
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=12)
best = dict(study.best_params, tree_method="hist", n_jobs=N_JOBS,
            scale_pos_weight=spw, eval_metric="aucpr")
tuned = xgb.XGBClassifier(**best).fit(Xtr, ytr)
score("XGBoost (Optuna-tuned)", tuned.predict_proba(Xte)[:, 1], time.time() - t0)
print("best params:", study.best_params)

## 4. Out-of-fold predictions for the ensembles

Four-fold time-series CV gives each base learner's predictions on data it
never trained on. These feed both the stacking meta-learner and the weight
search for the formula ensemble.

In [ ]:
t0 = time.time()
oof = {k: np.zeros(len(ytr)) for k in base}
oof_mask = np.zeros(len(ytr), dtype=bool)
last_val = None
for i, (tr_steps, va_steps) in enumerate(folds):
    tr, va = np.isin(ts_train, tr_steps), np.isin(ts_train, va_steps)
    for key, clf in make_base_models().items():
        clf.fit(Xtr[tr], ytr[tr])
        oof[key][va] = clf.predict_proba(Xtr[va])[:, 1]
    oof_mask |= va
    last_val = va
    print(f"oof fold {i + 1}/{len(folds)} done, elapsed {time.time() - t0:.0f}s")
oof_seconds = time.time() - t0

## 5. Formula-weighted ensemble

`p = w_xgb * p_xgb + w_lgbm * p_lgbm + w_cat * p_cat + w_rf * p_rf`, with
non-negative weights summing to 1, found by maximizing average precision on
the most recent out-of-fold slice.

In [ ]:
KEYS = list(base)
P = np.stack([oof[k][last_val] for k in KEYS], axis=1)


def neg_ap(w):
    w = np.abs(w) / np.abs(w).sum()
    return -average_precision_score(ytr[last_val], P @ w)


w = np.abs(minimize(neg_ap, np.ones(4) / 4, method="Nelder-Mead").x)
w = w / w.sum()
print("formula weights:", dict(zip(KEYS, np.round(w, 3))))
blend = sum(wi * test_probs[k] for wi, k in zip(w, KEYS))
score("Formula-weighted ensemble (XGB+LGBM+CAT+RF)", blend,
      sum(fit_seconds.values()) + oof_seconds)

## 6. Linear stacking ensemble

A logistic regression meta-learner (balanced class weights) is trained on
the out-of-fold predictions and applied to the full-train base learners'
test predictions.

In [ ]:
meta_X = np.stack([oof[k][oof_mask] for k in KEYS], axis=1)
meta_sc = StandardScaler().fit(meta_X)
meta = LogisticRegression(class_weight="balanced", max_iter=1000).fit(
    meta_sc.transform(meta_X), ytr[oof_mask])
stack = meta.predict_proba(meta_sc.transform(np.stack([test_probs[k] for k in KEYS], axis=1)))[:, 1]
score("Stacking ensemble (linear meta-learner)", stack,
      sum(fit_seconds.values()) + oof_seconds)
print("meta coefficients:", dict(zip(KEYS, np.round(meta.coef_[0], 3))))

## 7. GraphSAGE + attention GNN (3 hops)

Alternating GraphSAGE and GAT layers over the full transaction graph,
trained full-batch on the GPU (the graph is about 204k nodes and 234k
edges, small enough to fit without neighbor sampling). Loss uses
`pos_weight` for the class imbalance, and training stops early when the
training-set ROC-AUC stops improving. The test period is never used for
model selection.

In [ ]:
class SAGEAttnGNN(torch.nn.Module):
    def __init__(self, in_dim, hidden=128, num_layers=3, heads=4, dropout=0.3):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        dim = in_dim
        for i in range(num_layers):
            self.convs.append(SAGEConv(dim, hidden) if i % 2 == 0
                              else GATConv(dim, hidden // heads, heads=heads, dropout=dropout))
            dim = hidden
        self.out = torch.nn.Linear(dim, 1)
        self.dropout = dropout

    def forward(self, x, edge_index):
        for conv in self.convs:
            x = F.dropout(F.elu(conv(x, edge_index)), p=self.dropout, training=self.training)
        return self.out(x).squeeze(-1)


torch.manual_seed(0)
data = Data(x=torch.tensor(X), edge_index=torch.tensor(edge_index),
            y=torch.tensor(np.where(y < 0, 0, y), dtype=torch.float32)).to(DEVICE)
tr_t, te_t = torch.tensor(train_mask).to(DEVICE), torch.tensor(test_mask).to(DEVICE)
model = SAGEAttnGNN(data.num_node_features).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
pos_weight = torch.tensor([spw], device=DEVICE)


@torch.no_grad()
def probs_on(mask):
    model.eval()
    return torch.sigmoid(model(data.x, data.edge_index)[mask]).cpu().numpy()


t0, best_auc, best_state, bad = time.time(), -1.0, None, 0
MAX_EPOCHS, PATIENCE = 150, 15
for epoch in range(MAX_EPOCHS):
    model.train()
    opt.zero_grad()
    loss = F.binary_cross_entropy_with_logits(model(data.x, data.edge_index)[tr_t],
                                              data.y[tr_t], pos_weight=pos_weight)
    loss.backward()
    opt.step()
    train_auc = roc_auc_score(ytr, probs_on(tr_t))
    if train_auc > best_auc:
        best_auc, bad = train_auc, 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        bad += 1
    if epoch % 10 == 0:
        el = time.time() - t0
        print(f"epoch {epoch}/{MAX_EPOCHS} loss={loss.item():.4f} train_auc={train_auc:.4f} "
              f"elapsed={el:.0f}s eta={el / (epoch + 1) * (MAX_EPOCHS - epoch - 1):.0f}s")
    if bad >= PATIENCE:
        print("early stopping at epoch", epoch)
        break
model.load_state_dict(best_state)
score("GraphSAGE + Attention GNN (k=3)", probs_on(te_t), time.time() - t0)

## Results

All numbers are on the held-out test period (time steps 35 to 49). The
"vs NB" columns divide each metric by the Naive Bayes baseline.

In [ ]:
table = pd.DataFrame(RESULTS).T.round(4)
table["train_s"] = table["train_s"].round(1)
nb_row = table.loc["Naive Bayes (baseline)"]
table["ROC-AUC vs NB"] = (table["ROC-AUC"] / nb_row["ROC-AUC"]).round(3)
table["ROC-PR vs NB"] = (table["ROC-PR"] / nb_row["ROC-PR"]).round(3)
table["F1 vs NB"] = (table["F1"] / nb_row["F1"]).round(3)
print(table.to_string())
table